In [ ]:
import torch
from models.GPTModel import GPTModel
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "num_layers": 12,
    "num_heads": 12,
    "emb_dim": 768,
    "context_length": 1024,
    "dropout": 0.1,
    "num_classes": 2,
    'bias':False
}
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

In [ ]:
import tiktoken
from utils import generate
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    decoded = tokenizer.decode(token_ids.squeeze(0).tolist())
    return decoded

In [ ]:
context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = generate(
    model=model,
    idx=text_to_token_ids(context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M['context_length']
)
print(token_ids_to_text(token_ids, tokenizer))

In [ ]:
inputs = torch.tensor([[16833, 3626, 6100],
                       [40, 1107, 588]])
targets = torch.tensor([[3626, 6100, 345],
                        [1107, 588, 11311]])
with torch.no_grad():
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1)
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
target_probas_2 = probas[1, [0, 1, 2], targets[1]]

In [ ]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2), dim=-1))
print(log_probas)

In [ ]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

## 加载数据集

In [ ]:
file_path = "the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()
train_ratio = 0.9
split_idx = int(len(text) * train_ratio)
train_data = text[:split_idx]
val_data = text[split_idx:]

In [ ]:
print(len(train_data), len(val_data))

In [ ]:
from datasets.dataloader import create_dataloader
torch.manual_seed(123)
train_loader = create_dataloader(train_data, batch_size=2, max_length=GPT_CONFIG_124M['context_length'], 
                                 stride=GPT_CONFIG_124M['context_length'], shuffle=True, drop_last=True, num_workers=0)
val_loader = create_dataloader(val_data, batch_size=2, max_length=GPT_CONFIG_124M['context_length'], 
                               stride=GPT_CONFIG_124M['context_length'], shuffle=False, drop_last=True, num_workers=0)

In [ ]:
from chapter5.calc_loss import calc_loss, calc_loss_loader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    valid_loss = calc_loss_loader(val_loader, model, device)
print(f"train_loss: {train_loss:.3f}, valid_loss: {valid_loss:.3f}")

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, eval_iter)
        valid_loss = calc_loss_loader(val_loader, model, device, eval_iter)
    model.train()
    return train_loss, valid_loss

def generate_and_print_sample(model, tokenizer, start_context, device):
    model.eval()
    context_size = model.pos_embedding.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate(model, encoded, 50, context_size)
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace('\n', ' ')) 
    model.train()
    

def train_model_simple(model, train_loader, val_loader, optimizer, device, 
                       epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, step = 0, -1
    for epoch in range(epochs):
        model.train()
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            loss = calc_loss(inputs, targets, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += inputs.numel()
            step += 1

            if step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    track_tokens_seen.append(tokens_seen)
                    print(f"Epoch: {epoch}, Step: {step}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        generate_and_print_sample(model, tokenizer, start_context, device)
    return train_losses, val_losses, track_tokens_seen, model




In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)
epoch = 50
train_losses, val_losses, tokens_seen = train_model_simple(model, train_loader, val_loader, optimizer, device, epoch,
                                                           eval_freq=5, eval_iter=5, start_context="Every effort moves you", tokenizer=tokenizer)

## 控制随机性的解码策略

In [ ]:
model.to('cpu')
model.eval()
token_ids = generate(model, text_to_token_ids("Every effort moves you", tokenizer),
                          25,
                          context_size=GPT_CONFIG_124M['context_length'])
print(token_ids_to_text(token_ids, tokenizer))

In [3]:
def generate(model, idx, max_new_tokens, context_size,
             temperature=1.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        if top_k is not None:
            top_logits, _ = torch.topk(logits, k=top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, 
                                 torch.tensor(-float('Inf'), device=logits.device), 
                                 logits)
        if temperature > 0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.softmax(logits, dim=-1)
        if idx_next == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=-1)
    return idx

In [ ]:
token_ids = generate(
    model, 
    text_to_token_ids("Every effort moves you", tokenizer), 
    max_new_tokens=256,
    context_size=128,
    temperature=1.4,
    top_k=25 
)
print(token_ids_to_text(token_ids, tokenizer))

## 下载并加载预训练权重

In [ ]:
from gpt_download import download_and_load_gpt2
settings, params = download_and_load_gpt2(
    model_size="124M", models_dir="../models/GPT2-124M"
)

In [ ]:
from gpt_download import *
import torch
from models.GPTModel import GPTModel
from utils import *

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "num_layers": 12,
    "num_heads": 12,
    "emb_dim": 768,
    "context_length": 1024,
    "dropout": 0.1,
    "num_classes": 2,
    'bias':True
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dir = "../models/GPT2-124M/124M"
tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
settings = json.load(open(os.path.join(model_dir, "hparams.json"), "r", encoding="utf-8"))
params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)
model =  GPTModel(GPT_CONFIG_124M)
model.eval()
load_weights_into_gpt(model, params)
model.to(device)

GPTModel(
  (tok_embedding): Embedding(50257, 768)
  (pos_embedding): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (w_q): Linear(in_features=768, out_features=768, bias=True)
        (w_k): Linear(in_features=768, out_features=768, bias=True)
        (w_v): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attention): MultiHeadAttention(
        (w_q): Linear(in_featur

In [ ]:
from utils import generate, text_to_token_ids, token_ids_to_text
import tiktoken

torch.manual_seed(123)
tokenizer = tiktoken.get_encoding("gpt2")


Every effort moves you as far as the eye can see." (Gareth Fuller - A Modern American Crime Story) "That's right. I


In [18]:
token_ids = generate(
    model=model,
    idx=text_to_token_ids("Fuck you!", tokenizer).to(device),
    max_new_tokens=50,
    context_size=GPT_CONFIG_124M['context_length'],
    top_k=25,
    temperature=1.3
)
print(token_ids_to_text(token_ids, tokenizer))

Fuck you! No. No, NO!!!! Don't you fucking dare touch me!" (whisper)

She starts giggling, but eventually starts to get out of hand. The girl turns and kicks again, as though that's what would happen…
